# DGS DeepSEA on sciATAC1

This notebook trains and evaluates the DGS `DeepSEA` publication model on the sciATAC1 multi-label accessibility dataset. It keeps the original train / validate / test flow, but uses the DGS package API throughout.

Outline:

1. Configure paths, device, and runtime arguments.
2. Build HDF5-backed DataLoaders for sciATAC1.
3. Train `DeepSEA` with the DGS `Trainer`.
4. Evaluate task-level AUROC/AUPRC metrics.
5. Run optional motif interpretation with DGS `motif_enrich`.


In [1]:
from pathlib import Path
import argparse
import logging
import os
import subprocess
import sys
import time


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "DGS").exists() and (candidate / "Tutorials").exists():
            return candidate
    raise RuntimeError("Could not find the DeepGeSeq repository root from the current directory.")


# REPO_ROOT = find_repo_root(Path.cwd())
REPO_ROOT = find_repo_root("../../DeepGeSeq/")
TUTORIAL_DIR = REPO_ROOT / "Tutorials" / "3_DGS_seqAnalysis_sciATAC"
os.chdir(TUTORIAL_DIR)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.makedirs("Log", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(filename)s[line:%(lineno)d] %(levelname)s %(message)s",
    datefmt="%a, %d %b %Y %H:%M:%S",
    filename=time.strftime("./Log/log_DGS_deepsea.%m%d.%H:%M:%S.txt"),
    filemode="w",
    force=True,
)

parser = argparse.ArgumentParser()
parser.add_argument("--data", default=str(REPO_ROOT / "Tutorials" / "Dataset" / "Dataset.sciATAC1_train_test.h5"))
parser.add_argument("--gpu-device", dest="device_id", default="0")
parser.add_argument("--epochs", type=int, default=2)
parser.add_argument("--batch-size", type=int, default=128)
parser.add_argument("--num-workers", type=int, default=0)
parser.add_argument("--max-train-samples", type=int, default=0)
parser.add_argument("--max-eval-samples", type=int, default=0)
parser.add_argument("--max-explain-samples", type=int, default=2048)
parser.add_argument("--max-seqlets", type=int, default=2000)
parser.add_argument("--no-preload-data", action="store_true")
parser.add_argument("--attribution-method", default="deeplift_shap")

NOTEBOOK_ARGS = [
    "--data", str(REPO_ROOT / "Tutorials" / "Dataset" / "Dataset.sciATAC1_train_test.h5"),
    "--gpu-device", "0",
    "--epochs", "20",
    "--batch-size", "1280",
    "--max-explain-samples", "128",
    "--max-seqlets", "200",
]
args = parser.parse_args(NOTEBOOK_ARGS)
args.data = str(Path(args.data).expanduser().resolve())

if args.device_id.lower() in {"-1", "cpu", "none"}:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = args.device_id

logging.info(args)
print(f"Repository root: {REPO_ROOT}")
print(f"Tutorial directory: {TUTORIAL_DIR}")
print(f"Dataset: {args.data}")


Repository root: /autodl-fs/data/DeepGeSeq
Tutorial directory: /autodl-fs/data/DeepGeSeq/Tutorials/3_DGS_seqAnalysis_sciATAC
Dataset: /autodl-fs/data/Tutorials/Dataset/Dataset.sciATAC1_train_test.h5


In [2]:
import h5py
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Subset

from DGS.DL.Trainer import Trainer
from DGS.Model.Publications import DeepSEA
from DGS.DL.Evaluator import calculate_classification_metrics
from DGS.DL.Explain import motif_enrich

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


Using device: cuda


## Prepare dataset

1. unpack the h5file datasets
2. generate the DataLoader


In [ ]:
# unpack datasets
h5file = h5py.File(args.data, 'r')
anno = h5file["annotation"][:]
x_train = h5file["train_data"][:].astype(np.float32)
y_train = h5file["train_label"][:].astype(np.float32)
x_val = h5file["val_data"][:].astype(np.float32)
y_val = h5file["val_label"][:].astype(np.float32)
x_test = h5file["test_data"][:].astype(np.float32)
y_test = h5file["test_label"][:].astype(np.float32)
h5file.close()

# unpack anno
sequence_length = int(x_train.shape[-1])
n_tasks = int(anno.shape[0])
task_name = pd.Index(
    [value.decode() if isinstance(value, bytes) else str(value) for value in anno[:, 0]],
    name="task",
)

# define data loader
batch_size = args.batch_size
train_dataset = list(zip(x_train, y_train))
validate_dataset = list(zip(x_val, y_val))
test_dataset = list(zip(x_test, y_test))
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True, num_workers=2, drop_last=False, pin_memory=True)
validate_loader = DataLoader(validate_dataset, batch_size=batch_size,
                             shuffle=False, num_workers=2, drop_last=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2, drop_last=False, pin_memory=True)

print(f"sequence_length={sequence_length}, n_tasks={n_tasks}")
print(f"batch_size={batch_size}")
print(f"train batches: {len(train_loader):,}")
print(f"validate batches: {len(validate_loader):,}")
print(f"test batches: {len(test_loader):,}")


## Define DeepSEA Model

Initialize the DGS `DeepSEA` architecture for the sciATAC1 sequence length and 85 genomic accessibility tasks.


In [16]:
MODEL_NAME = "DeepSEA"

model = DeepSEA(sequence_length=sequence_length, n_genomic_features=n_tasks)
model


DeepSEA(
  (conv_net): Sequential(
    (0): Conv1d(4, 320, kernel_size=(8,), stride=(1,))
    (1): ReLU(inplace=True)
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Dropout(p=0.2, inplace=False)
    (4): Conv1d(320, 480, kernel_size=(8,), stride=(1,))
    (5): ReLU(inplace=True)
    (6): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.2, inplace=False)
    (8): Conv1d(480, 960, kernel_size=(8,), stride=(1,))
    (9): ReLU(inplace=True)
    (10): Dropout(p=0.5, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=26880, out_features=85, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=85, out_features=85, bias=True)
    (3): Sigmoid()
  )
)

In [17]:
checkpoint_dir = Path("Log") / "checkpoints" / "DeepSEA_sciATAC1"
optimizer = Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

trainer = Trainer(
    model,
    criterion,
    optimizer,
    device,
    checkpoint_dir=checkpoint_dir,
    patience=10,
    use_tensorboard=False,
)
checkpoint_dir


PosixPath('Log/checkpoints/DeepSEA_sciATAC1')

## Train Model

As in the original tutorial, this example trains for 2 epochs. For a quick smoke test, add `--max-train-samples` and `--max-eval-samples` to `NOTEBOOK_ARGS` above.


In [18]:
# train
metrics = trainer.train(train_loader, validate_loader, epochs=args.epochs)
metrics


Epoch 19: 100%|██████████| 366/366 [00:28<00:00, 12.94it/s, loss=0.0547]


TrainerMetrics(train_losses=[0.07706159935460065, 0.06389360933925936, 0.06359507406099899, 0.06342968735538546, 0.06330620145292881, 0.06316404126055254, 0.06290802547881186, 0.06230961593488852, 0.06183763556793088, 0.061403006294874544, 0.06084156150374908, 0.06007590308744725, 0.05920539142899826, 0.05840442744923419, 0.05780362890514194, 0.05717177907910829, 0.05641909214398248, 0.05578371606604323, 0.05516534830988105, 0.054659036726085214], val_losses=[0.06442531230834013, 0.06425022736924593, 0.06370426474210343, 0.06335217074129751, 0.06325297538048583, 0.06316691620753763, 0.062654439482044, 0.061948903434263555, 0.061489475962242794, 0.060803021299220175, 0.060365977040568335, 0.059245135777635, 0.05836739266912142, 0.05770997666433209, 0.05691709693093769, 0.05639480882899357, 0.05585670711401382, 0.05537252476225134, 0.054402332590088816, 0.05382916215257567], train_metrics=[], val_metrics=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

## Evaluate Model

Load the best checkpoint when it exists, then compute task-level classification metrics on the sciATAC1 test split.


In [19]:
best_checkpoint = checkpoint_dir / "best_model.pt"
if best_checkpoint.exists():
    trainer.load_checkpoint(best_checkpoint, load_optimizer=False)
    print(f"Loaded best checkpoint: {best_checkpoint}")
else:
    print(f"No best checkpoint found at {best_checkpoint}; evaluating the current model state.")
model = trainer.model


Loaded best checkpoint: Log/checkpoints/DeepSEA_sciATAC1/best_model.pt


In [20]:
# evaluate test-set
_, _, test_predictions, test_targets = trainer.validate(test_loader, return_predictions=True)

metric_df = calculate_classification_metrics(
    test_targets.cpu().numpy(),
    test_predictions.cpu().numpy(),
)
metric_df.index = task_name
metric_path = Path("Metric.DeepSEA_sciATAC1.csv")
metric_df.to_csv(metric_path)
print(f"Saved metrics to {metric_path}")
metric_df


Saved metrics to Metric.DeepSEA_sciATAC1.csv


,auroc,auprc,f1,accuracy
task,,,,
b'10_1',0.796505,0.090625,0.0,0.987953
b'11_1',0.875360,0.089322,0.0,0.988171
b'11_2',0.878839,0.092953,0.0,0.988171
b'11_3',0.861993,0.084481,0.0,0.988304
b'11_4',0.859848,0.071415,0.0,0.987897
...,...,...,...,...
b'8_1',0.817530,0.062366,0.0,0.988252
b'8_2',0.813478,0.058648,0.0,0.988103
b'9_1',0.794738,0.044085,0.0,0.988214


In [21]:
metric_df.sort_values("auroc", ascending=False).head(10)


,auroc,auprc,f1,accuracy
task,,,,
b'12_5',0.998352,0.790012,0.824728,0.995658
b'30_2',0.996861,0.698630,0.438736,0.990652
b'30_3',0.986515,0.458557,0.124160,0.988291
b'26_3',0.963613,0.274232,0.044191,0.988526
b'30_4',0.959097,0.230808,0.005585,0.987812
b'30_1',0.889003,0.089571,0.000000,0.988321
b'17_4',0.886766,0.093691,0.000000,0.988582
b'12_4',0.880951,0.080460,0.000000,0.988304
b'11_2',0.878839,0.092953,0.000000,0.988171


## Model Interpretation

DGS `motif_enrich` calculates sequence attributions and runs TF-MoDISco-lite when optional explain dependencies are installed. The default cell uses a capped subset of test sequences for a practical tutorial run; set `--max-explain-samples 0` in `NOTEBOOK_ARGS` to use the full test split.


In [11]:
explain_dir = Path("Explain") / "DeepSEA_sciATAC1"
explain_dir.mkdir(parents=True, exist_ok=True)

if args.max_explain_samples and args.max_explain_samples > 0:
    n_explain = min(args.max_explain_samples, len(test_dataset))
    explain_dataset = Subset(test_dataset, range(n_explain))
else:
    explain_dataset = test_dataset

try:
    motif_file = motif_enrich(
        model,
        explain_dataset,
        target=0,
        output_dir=str(explain_dir),
        max_seqlets=args.max_seqlets,
        device=device,
        batch_size=args.batch_size,
        method=args.attribution_method,
    )
    print(motif_file)
except (RuntimeError, FileNotFoundError, subprocess.CalledProcessError) as exc:
    motif_file = None
    print(f"Skipped motif enrichment: {type(exc).__name__}: {exc}")


Skipped motif enrichment: RuntimeError: Output 0 of BackwardHookFunctionBackward is a view and is being modified inplace. This view was created inside a custom Function (or because an input was returned as-is) and the autograd logic to handle view+inplace would override the custom backward associated with the custom Function, leading to incorrect gradients. This behavior is forbidden. You can fix this by cloning the output of the custom Function.


In [ ]:
motif_path = explain_dir / "motifs.txt"
if motif_path.exists():
    print("\n".join(motif_path.read_text().splitlines()[:20]))
else:
    print(f"No motifs file found at {motif_path}.")


## Exercise And Extension

Compare `Basset` and `DeepSEA` by sorting their saved metric CSV files by AUROC or AUPRC, then repeat motif enrichment for a different target index from `task_name`.
